# South Africa JUSTICE project notebook — integrated project workflow

**Actor:** South Africa (`zaf`)  
**Core argument:** South Africa supports mitigation, but rejects equal mitigation obligations if the transition burden is not fairly financed.

This notebook is organised by the **project approach**, not by separate assignment numbers. Under each project step, it says which assignment provides the input or result.

The workflow is:

1. Frame South Africa's decision problem  
2. Build the optimisation setup  
3. Generate candidate policies through MOEA  
4. Check search quality and build a reference set  
5. Understand trade-offs and select policy candidates  
6. Re-evaluate the preferred policy for full time series  
7. Analyse South Africa-specific transition burden  
8. Test robustness and satisficing  
9. Compare with a rival framing  
10. Translate results into policy advice

> Do not run the whole notebook at once. Some JUSTICE and optimisation cells are slow. Run section by section.

## Project workflow overview

Use this table as the map for the whole project. The headings in this notebook follow the **project approach**, while the final column shows which assignment provides the material for each step.

| Project step | Main question | What you need from the assignments | South Africa use |
|---|---|---|---|
| **1. Frame South Africa’s decision problem** | What does the mandate mean in model terms? | **Assignment 4:** mandate translation, XLRM, welfare lens, key region, key variables, satisficing criterion | Define `zaf`, Prioritarian welfare, and burden as `abatement_cost / gross_economic_output` |
| **2. Build the optimisation setup** | How do we turn the framing into an optimisation problem? | **Assignment 4:** RBF decision variables, scenario choice, objectives, epsilons, `config_student.json` | Set up adaptive mitigation policies rather than fixed equal obligations |
| **3. Generate candidate policies through MOEA** | What policies does the optimiser find? | **Assignment 5:** local MOEA run, seeds, NFE, ensemble sample, Pareto archives | Generate candidate adaptive mitigation policies for South Africa’s position |
| **4. Check search quality and build a reference set** | Did the optimiser search well enough? | **Assignment 6:** convergence plots, epsilon progress, combined reference set | Make sure your policy choice is not based on one random or weak optimiser run |
| **5. Understand trade-offs and select candidate policies** | What trade-offs exist between climate safety and transition burden? | **Assignment 7:** Pareto plots, parallel coordinates, selected policies | Compare climate-safe, low-cost, and compromise policies; choose a South Africa-preferred option |
| **6. Re-evaluate preferred policy for full time series** | What happens over time under the chosen policy? | **Assignment 7/8 + extra analysis:** re-run selected policy and extract time series | Get `abatement_cost`, `gross_economic_output`, `economic_damage`, ECR and temperature over time |
| **7. South Africa-specific burden analysis** | Is the transition burden disproportionate for South Africa? | **Extra actor-specific analysis using re-evaluation outputs** | Calculate `zaf_abatement_cost / zaf_gross_economic_output` and compare with developed regions |
| **8. Robustness and satisficing** | Does the preferred policy stay acceptable under uncertainty? | **Assignment 8:** robustness testing across scenarios and/or climate ensembles | Test whether South Africa’s minimum acceptable outcomes are met often enough |
| **9. Rival framing** | Does the conclusion change under another welfare lens? | **Assignment 8 / extra:** compare Prioritarian with Utilitarian or another rival framing | Show what a global-efficiency framing hides about unequal transition costs |
| **10. Final policy advice** | What should South Africa argue for? | Combine all previous steps | Support ambitious mitigation only if paired with dedicated Just Transition finance |

**Core logic:** Assignment 4 frames the problem; Assignment 5 generates policies; Assignment 6 checks whether the search is credible; Assignment 7 helps interpret and select policies; Assignment 8 tests robustness. The South Africa-specific contribution is the burden analysis relative to GDP and the rival-framing comparison.


## 0. Setup and file locations

**Purpose:** prepare imports, paths, constants, and region indices used throughout the project.

This setup is used by all later project steps.

In [1]:
# import os
# import sys
# import json
# from pathlib import Path
# import warnings
#
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
#
# warnings.filterwarnings("ignore")
#
# # Try to detect notebook directory. In VS Code notebooks, __vsc_ipynb_file__ is available.
# try:
#     _NOTEBOOK_DIR = Path(__vsc_ipynb_file__).resolve().parent
# except NameError:
#     _NOTEBOOK_DIR = Path.cwd().resolve()
#
# # Expected course/project structure: notebook is in assignments_ema, JUSTICE-main is next to it.
# _JUSTICE_ROOT = (_NOTEBOOK_DIR / "../JUSTICE-main").resolve()
# _CONFIG_DIR = (_NOTEBOOK_DIR / "../config").resolve()
# RESULTS_DIR = (_NOTEBOOK_DIR / "results").resolve()
# PLOTS_DIR = (_NOTEBOOK_DIR / "plots").resolve()
#
# for path in [_CONFIG_DIR, RESULTS_DIR, PLOTS_DIR]:
#     path.mkdir(parents=True, exist_ok=True)
#
# if str(_JUSTICE_ROOT) not in sys.path:
#     sys.path.insert(0, str(_JUSTICE_ROOT))
#
# # Some JUSTICE imports assume the working directory is JUSTICE-main.
# os.chdir(_JUSTICE_ROOT)
#
# from justice.model import JUSTICE
# from justice.util.enumerations import WelfareFunction, Scenario
#
# print("Notebook dir:", _NOTEBOOK_DIR)
# print("JUSTICE root:", _JUSTICE_ROOT)
# print("Config dir:", _CONFIG_DIR)
# print("Results dir:", RESULTS_DIR)

In [2]:
import os
import sys
import json
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# Try to detect notebook directory. In VS Code notebooks, __vsc_ipynb_file__ is available.
try:
    _NOTEBOOK_DIR = Path(__vsc_ipynb_file__).resolve().parent
except NameError:
    _NOTEBOOK_DIR = Path.cwd().resolve()

# Find repo root: the folder that contains JUSTICE-main and config
_REPO_ROOT = _NOTEBOOK_DIR
for candidate in [_NOTEBOOK_DIR] + list(_NOTEBOOK_DIR.parents):
    if (candidate / "JUSTICE-main").exists() and (candidate / "config").exists():
        _REPO_ROOT = candidate
        break

_JUSTICE_ROOT = (_REPO_ROOT / "JUSTICE-main").resolve()
_CONFIG_DIR = (_REPO_ROOT / "config").resolve()

# Project/group folder
_PROJECT_DIR = (_REPO_ROOT / "A- Project G15").resolve()

# Important: keep all generated project files inside A- Project G15
RESULTS_DIR = (_PROJECT_DIR / "results").resolve()
PLOTS_DIR = (_PROJECT_DIR / "plots").resolve()

# Config folder should already exist; results and plots can be created
for path in [RESULTS_DIR, PLOTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

if str(_JUSTICE_ROOT) not in sys.path:
    sys.path.insert(0, str(_JUSTICE_ROOT))

# Some JUSTICE imports assume the working directory is JUSTICE-main.
# This does NOT affect RESULTS_DIR/PLOTS_DIR because they are absolute Path objects.
os.chdir(_JUSTICE_ROOT)

from justice.model import JUSTICE
from justice.util.enumerations import WelfareFunction, Scenario

print("Notebook dir:", _NOTEBOOK_DIR)
print("Repo root:", _REPO_ROOT)
print("Project dir:", _PROJECT_DIR)
print("JUSTICE root:", _JUSTICE_ROOT)
print("Config dir:", _CONFIG_DIR)
print("Results dir:", RESULTS_DIR)
print("Plots dir:", PLOTS_DIR)

assert RESULTS_DIR == (_PROJECT_DIR / "results").resolve()

Notebook dir: /Users/wesselmuntendam/Model Based Decision Making/epa141a/A- Project G15
Repo root: /Users/wesselmuntendam/Model Based Decision Making/epa141a
Project dir: /Users/wesselmuntendam/Model Based Decision Making/epa141a/A- Project G15
JUSTICE root: /Users/wesselmuntendam/Model Based Decision Making/epa141a/JUSTICE-main
Config dir: /Users/wesselmuntendam/Model Based Decision Making/epa141a/config
Results dir: /Users/wesselmuntendam/Model Based Decision Making/epa141a/A- Project G15/results
Plots dir: /Users/wesselmuntendam/Model Based Decision Making/epa141a/A- Project G15/plots


In [3]:
# Basic model dimensions used in Assignment 4 and later optimisation.
# These are the standard values used in the assignment setup.
N_REGIONS = 57
N_INPUTS = 2
N_RBFS = N_INPUTS + 2
START_YEAR = 2015
END_YEAR = 2300
TIMESTEP = 1
DATA_TIMESTEP = 5

# We get the exact region list from JUSTICE. This is light compared with full optimisation.
JUSTICE.hard_reset()
_model_probe = JUSTICE(
    start_year=START_YEAR,
    end_year=END_YEAR,
    timestep=TIMESTEP,
    scenario=2,
    climate_ensembles=1,
    stochastic_run=False,
    social_welfare_function=WelfareFunction.PRIORITARIAN,
)

REGION_LIST = list(_model_probe.data_loader.REGION_LIST)
N_REGIONS = len(REGION_LIST)
N_TIMESTEPS = len(_model_probe.time_horizon.model_time_horizon)
zaf_idx = REGION_LIST.index("zaf")

print("Number of regions:", N_REGIONS)
print("Number of timesteps:", N_TIMESTEPS)
print("South Africa index:", zaf_idx)
print("First 10 regions:", REGION_LIST[:10])

Number of regions: 57
Number of timesteps: 286
South Africa index: 56
First 10 regions: ['arg', 'aus', 'aut', 'bel', 'bgr', 'blt', 'bra', 'can', 'chl', 'chn']


## 1. Frame South Africa's decision problem

**Project purpose:** translate the South Africa mandate into model terms.

**Main source:** Assignment 4 — problem formulation.  
**Later use:** every later analysis should connect back to this framing.

South Africa's position in plain language:

> South Africa supports climate mitigation, but rejects equal obligations that ignore coal dependence, development level, transition capacity, and the economic burden of decarbonisation.

The key modelling idea is therefore **relative transition burden**, not only absolute emissions or global welfare.

### 1.1 Mandate translated to model terms

| Mandate element | Model translation | Where used later |
|---|---|---|
| South Africa is coal-dependent | Focus on `zaf` and abatement costs | Re-evaluation and burden analysis |
| Rejects equal obligations regardless of wealth | Compare costs as share of GDP/output | South Africa burden table |
| Needs Just Transition finance | Identify policies where mitigation creates high relative burden | Policy advice |
| Supports Africa's demands if finance is available | Include climate risk and damage outcomes | Pareto and robustness analysis |
| Welfare lens gives extra weight to worse-off regions | Use Prioritarian welfare as main framing | Optimisation setup |
| Mandate says express everything relative to output | Use `abatement_cost / gross_economic_output` | Main actor-specific metric |

Core actor-specific metric:

```python
abatement_burden = abatement_cost / gross_economic_output
```

In [4]:
ACTOR = "South Africa"
KEY_REGION = "zaf"
PREFERRED_WELFARE = WelfareFunction.PRIORITARIAN
RIVAL_WELFARE = WelfareFunction.UTILITARIAN

south_africa_actor_frame = {
    "actor": ACTOR,
    "key_region": KEY_REGION,
    "preferred_welfare_function": PREFERRED_WELFARE.name,
    "rival_welfare_function": RIVAL_WELFARE.name,
    "core_metric": "abatement_cost / gross_economic_output",
    "main_claim": (
        "South Africa supports mitigation, but equal obligations are not equally fair "
        "when transition costs are unequal relative to economic capacity."
    ),
}

south_africa_actor_frame

{'actor': 'South Africa',
 'key_region': 'zaf',
 'preferred_welfare_function': 'PRIORITARIAN',
 'rival_welfare_function': 'UTILITARIAN',
 'core_metric': 'abatement_cost / gross_economic_output',
 'main_claim': 'South Africa supports mitigation, but equal obligations are not equally fair when transition costs are unequal relative to economic capacity.'}

### 1.2 XLRM framework for South Africa

This is the filled XLRM that connects the political mandate to model choices.

- **X — Uncertainties:** things South Africa cannot control.
- **L — Levers:** what the policy/search can change.
- **R — Relationships:** how JUSTICE turns policy into outcomes.
- **M — Metrics:** how we judge policy performance.

In [5]:
south_africa_xlrm = {
    "X — Uncertainties": [
        "Future socioeconomic development pathways",
        "Climate sensitivity and climate response",
        "Regional climate damages affecting South Africa",
        "Future abatement costs",
        "Future backstop costs",
        "Availability of transition funds",
        "Damage function scale factor δ: uncertainty in how strongly temperature creates economic damages",
        "Pure rate of time preference ρ: normative uncertainty about how much future welfare counts",
        "Elasticity of marginal utility η: normative uncertainty about how strongly poorer regions are weighted",
    ],

    "L — Policy levers": [
        "Emission control rate (ECR / μ)",
        "Savings rate: share of net output invested rather than consumed",
        "Adaptive emission reduction policy: adjusts ECR over time based on climate conditions",
    ],

    "R — Model relationships": [
        "GDP drives baseline emissions",
        "ECR reduces emissions",
        "Emissions determine global and regional temperature through FaIR",
        "Temperature creates economic damages",
        "ECR creates abatement costs",
        "Damages and abatement costs reduce consumption",
        "Consumption is aggregated through a social welfare function",
    ],

    "M — Performance measures": [
        "Welfare (maximize)",
        "Global temperature (minimize)",
        "Damage burden: economic_damage / gross_economic_output (minimize)",
        "Mitigation burden: abatement_cost / gross_economic_output (minimize)",
        "Consumption_per_capita (maximize)",
        "Gross_economic_output (maximize)",
        "Net output share: net_economic_output / gross_economic_output (maximize)",
        "fraction_above_threshold_max (minimize)",

    ],
}

south_africa_xlrm

{'X — Uncertainties': ['Future socioeconomic development pathways',
  'Climate sensitivity and climate response',
  'Regional climate damages affecting South Africa',
  'Future abatement costs',
  'Future backstop costs',
  'Availability of transition funds',
  'Damage function scale factor δ: uncertainty in how strongly temperature creates economic damages',
  'Pure rate of time preference ρ: normative uncertainty about how much future welfare counts',
  'Elasticity of marginal utility η: normative uncertainty about how strongly poorer regions are weighted'],
 'L — Policy levers': ['Emission control rate (ECR / μ)',
  'Savings rate: share of net output invested rather than consumed',
  'Adaptive emission reduction policy: adjusts ECR over time based on climate conditions'],
 'R — Model relationships': ['GDP drives baseline emissions',
  'ECR reduces emissions',
  'Emissions determine global and regional temperature through FaIR',
  'Temperature creates economic damages',
  'ECR create

In [6]:
# XLRM → EMA/JUSTICE mapping for South Africa
ema_mapping = pd.DataFrame([
    # -------------------------
    # X — Uncertainties
    # -------------------------
    (
        "X — Uncertainties",
        "Future socioeconomic development pathways",
        "CategoricalParameter / Scenario",
        "em_model.uncertainties",
        "Usually represented by ssp_rcp_scenario"
    ),
    (
        "X — Uncertainties",
        "Climate sensitivity and climate response",
        "Climate ensemble / FaIR ensemble member",
        "model constant or experiment setting",
        "Represents climate response uncertainty; not a policy lever"
    ),
    (
        "X — Uncertainties",
        "Regional climate damages affecting South Africa",
        "Model uncertainty / scenario outcome",
        "model input or post-run analysis",
        "Captured through temperature, damage function, and regional damage outputs"
    ),
    (
        "X — Uncertainties",
        "Future abatement costs",
        "Model uncertainty / parameter",
        "model input or experiment setting",
        "Relevant because abatement cost differs by region and energy system"
    ),
    (
        "X — Uncertainties",
        "Future backstop costs",
        "RealParameter or scenario assumption",
        "em_model.uncertainties or experiment setting",
        "Lower backstop_cost means cheaper clean technology"
    ),
    (
        "X — Uncertainties",
        "Availability of transition funds",
        "Scenario assumption",
        "experiment setting / interpretation",
        "Can be represented indirectly as lower effective abatement or backstop costs"
    ),
    (
        "X — Uncertainties",
        "Damage function scale factor δ",
        "RealParameter [0.5, 2.0]",
        "em_model.uncertainties",
        "Uncertainty in how strongly temperature creates economic damages"
    ),
    (
        "X — Uncertainties",
        "Pure rate of time preference ρ",
        "RealParameter [0.001, 0.030]",
        "em_model.uncertainties",
        "Normative uncertainty about how much future welfare counts"
    ),
    (
        "X — Uncertainties",
        "Elasticity of marginal utility η",
        "RealParameter [0.5, 2.5]",
        "em_model.uncertainties",
        "Normative uncertainty about how strongly poorer regions are weighted"
    ),

    # -------------------------
    # L — Levers
    # -------------------------
    (
        "L — Policy levers",
        "Emission control rate (ECR / μ)",
        "Policy variable",
        "em_model.levers, through RBF policy",
        "Main mitigation lever: determines how much emissions are reduced"
    ),
    (
        "L — Policy levers",
        "Savings rate",
        "RealParameter or fixed/endogenous model setting",
        "em_model.levers or model setting",
        "Only include as a lever if your group explicitly varies it"
    ),
    (
        "L — Policy levers",
        "Adaptive emission reduction policy",
        "RBF centers, radii, and weights",
        "em_model.levers",
        "Technical decision variables that determine ECR over time"
    ),

    # -------------------------
    # M — Direct EMA outcomes
    # -------------------------
    (
        "M — Performance measures",
        "Welfare",
        "ScalarOutcome",
        "em_model.outcomes",
        "Can be minimized or maximized depending on wrapper/sign convention"
    ),
    (
        "M — Performance measures",
        "fraction_above_threshold_max",
        "ScalarOutcome MINIMIZE",
        "em_model.outcomes or post-processing",
        "Temperature safety/risk measure"
    ),
    (
        "M — Performance measures",
        "Global temperature",
        "Time series / scalar summary",
        "model output or post-processing",
        "Usually minimize final-year or max temperature"
    ),

    # -------------------------
    # M — Post-processing measures
    # -------------------------
    (
        "M — Performance measures",
        "Damage burden",
        "Calculated measure",
        "post-processing",
        "economic_damage / gross_economic_output; minimize"
    ),
    (
        "M — Performance measures",
        "Mitigation burden",
        "Calculated measure",
        "post-processing",
        "abatement_cost / gross_economic_output; minimize"
    ),
    (
        "M — Performance measures",
        "Consumption_per_capita",
        "Model output",
        "model output / post-processing",
        "Material consumption per person; maximize"
    ),
    (
        "M — Performance measures",
        "Gross_economic_output",
        "Model output",
        "model output / post-processing",
        "Mostly used as denominator/context; maximize if included directly"
    ),
    (
        "M — Performance measures",
        "Net output share",
        "Calculated measure",
        "post-processing",
        "net_economic_output / gross_economic_output; maximize"
    ),
], columns=["XLRM", "Variable", "EMA/JUSTICE type", "Assignment to", "Notes"])

print("\nXLRM → EMA/JUSTICE mapping:")
display(ema_mapping)


XLRM → EMA/JUSTICE mapping:


,XLRM,Variable,EMA/JUSTICE type,Assignment to,Notes
0,X — Uncertainties,Future socioeconomic development pathways,CategoricalParameter / Scenario,em_model.uncertainties,Usually represented by ssp_rcp_scenario
1,X — Uncertainties,Climate sensitivity and climate response,Climate ensemble / FaIR ensemble member,model constant or experiment setting,Represents climate response uncertainty; not a...
2,X — Uncertainties,Regional climate damages affecting South Africa,Model uncertainty / scenario outcome,model input or post-run analysis,"Captured through temperature, damage function,..."
3,X — Uncertainties,Future abatement costs,Model uncertainty / parameter,model input or experiment setting,Relevant because abatement cost differs by reg...
4,X — Uncertainties,Future backstop costs,RealParameter or scenario assumption,em_model.uncertainties or experiment setting,Lower backstop_cost means cheaper clean techno...
5,X — Uncertainties,Availability of transition funds,Scenario assumption,experiment setting / interpretation,Can be represented indirectly as lower effecti...
6,X — Uncertainties,Damage function scale factor δ,"RealParameter [0.5, 2.0]",em_model.uncertainties,Uncertainty in how strongly temperature create...
7,X — Uncertainties,Pure rate of time preference ρ,"RealParameter [0.001, 0.030]",em_model.uncertainties,Normative uncertainty about how much future we...
8,X — Uncertainties,Elasticity of marginal utility η,"RealParameter [0.5, 2.5]",em_model.uncertainties,Normative uncertainty about how strongly poore...
9,L — Policy levers,Emission control rate (ECR / μ),Policy variable,"em_model.levers, through RBF policy",Main mitigation lever: determines how much emi...


### 1.3 Satisficing criterion for South Africa

A satisficing criterion is the **minimum acceptable outcome** for the actor.

For South Africa, a policy does not need to be globally perfect. It needs to be politically and economically acceptable:

1. Climate risk should not be too high.
2. South Africa's transition burden should not be disproportionate relative to economic capacity.

The exact threshold can be adjusted later once real outputs are available.

In [7]:
SATISFICING = {
    # Example thresholds. These are working assumptions, not final claims.
    "fraction_above_threshold_max": 0.25,
    "zaf_mean_abatement_burden_max": 0.05,  # 5% of gross output as working threshold
    "zaf_burden_comparison_rule": (
        "South Africa's abatement burden should not greatly exceed the burden in developed reference regions."
    ),
}

SATISFICING

{'fraction_above_threshold_max': 0.25,
 'zaf_mean_abatement_burden_max': 0.05,
 'zaf_burden_comparison_rule': "South Africa's abatement burden should not greatly exceed the burden in developed reference regions."}

In [8]:
SATISFICING = {
    # Climate effectiveness criterion
    # A policy should not leave too much probability/risk of exceeding the chosen temperature threshold.
    "fraction_above_threshold_max": 0.25,

    # South Africa fairness criterion
    # Working assumption: South Africa's average mitigation burden should stay below 5% of gross output.
    # This is not yet a final claim; it needs to be justified with literature or sensitivity testing.
    "zaf_mean_abatement_burden_max": 0.05,

    # Comparative fairness criterion
    # South Africa should not carry a substantially higher mitigation burden than developed reference regions.
    "zaf_burden_comparison_rule": (
        "South Africa's abatement burden should not greatly exceed the burden in developed reference regions."
    ),

    # Welfare / living standards criterion
    # Climate policy should not make South Africa's future material living standards worse than under no policy.
    "zaf_consumption_per_capita_rule": (
        "South Africa's future consumption_per_capita should be higher than, or at least not lower than, "
        "the no-policy baseline in the long run."
    ),

    # Net economic output criterion
    # A policy should preserve South Africa's net output relative to gross output.
    "zaf_net_output_share_rule": (
        "South Africa's net_economic_output / gross_economic_output should remain sufficiently high, "
        "meaning that damages and abatement costs do not absorb an unacceptable share of output."
    ),

    # Damage burden criterion
    # No policy is also unacceptable if climate damages become too large.
    "zaf_damage_burden_rule": (
        "South Africa's economic_damage / gross_economic_output should be reduced compared with the no-policy baseline."
    ),

    # Welfare criterion
    # Since South Africa's mandate is justice-oriented, prioritarian welfare is preferred over purely utilitarian welfare.
    "welfare_rule": (
        "Among policies satisfying the South Africa-specific burden constraints, prefer policies with higher prioritarian welfare."
    ),
}

SATISFICING

{'fraction_above_threshold_max': 0.25,
 'zaf_mean_abatement_burden_max': 0.05,
 'zaf_burden_comparison_rule': "South Africa's abatement burden should not greatly exceed the burden in developed reference regions.",
 'zaf_consumption_per_capita_rule': "South Africa's future consumption_per_capita should be higher than, or at least not lower than, the no-policy baseline in the long run.",
 'zaf_net_output_share_rule': "South Africa's net_economic_output / gross_economic_output should remain sufficiently high, meaning that damages and abatement costs do not absorb an unacceptable share of output.",
 'zaf_damage_burden_rule': "South Africa's economic_damage / gross_economic_output should be reduced compared with the no-policy baseline.",
 'welfare_rule': 'Among policies satisfying the South Africa-specific burden constraints, prefer policies with higher prioritarian welfare.'}

In [9]:
def south_africa_satisficing(metrics, thresholds=SATISFICING):
    # Simple satisficing rule for South Africa.
    # Expected keys in metrics: fraction_above_threshold, zaf_mean_abatement_burden.
    return (
        metrics["fraction_above_threshold"] <= thresholds["fraction_above_threshold_max"]
        and metrics["zaf_mean_abatement_burden"] <= thresholds["zaf_mean_abatement_burden_max"]
    )

### 1.4 Optional initial model intuition: BAU and welfare-function comparison

**Source:** earlier project notebook.  
**Purpose:** optional background intuition before optimisation.

This section is not the main MOEA analysis. It can help you explain why welfare framing matters and why South Africa-specific outputs are needed.

In [10]:
# Optional: compare welfare functions under BAU (no abatement).
# This can take some time. You can skip this cell if you only need the optimisation workflow.

RUN_OPTIONAL_BAU_WELFARE_COMPARISON = True

if RUN_OPTIONAL_BAU_WELFARE_COMPARISON:
    welfare_functions = {
        "Utilitarian": WelfareFunction.UTILITARIAN,
        "Prioritarian": WelfareFunction.PRIORITARIAN,
        "Sufficientarian": WelfareFunction.SUFFICIENTARIAN,
    }

    results_wf = {}
    for wf_name, wf in welfare_functions.items():
        JUSTICE.hard_reset()
        model = JUSTICE(
            start_year=START_YEAR,
            end_year=END_YEAR,
            timestep=TIMESTEP,
            scenario=2,
            climate_ensembles=1,
            stochastic_run=False,
            social_welfare_function=wf,
        )
        ecr_bau = np.zeros((len(model.data_loader.REGION_LIST), len(model.time_horizon.model_time_horizon)))
        model.run(emission_control_rate=ecr_bau, endogenous_savings_rate=True)
        data = model.evaluate()
        results_wf[wf_name] = data["welfare"]

    results_wf
else:
    print("Skipping optional BAU welfare comparison. Set RUN_OPTIONAL_BAU_WELFARE_COMPARISON = True to run.")

In [11]:
RUN_OPTIONAL_BAU_WELFARE_COMPARISON = True

if RUN_OPTIONAL_BAU_WELFARE_COMPARISON:
    welfare_functions = {
        "Utilitarian": WelfareFunction.UTILITARIAN,
        "Prioritarian": WelfareFunction.PRIORITARIAN,
        "Sufficientarian": WelfareFunction.SUFFICIENTARIAN,
    }

    results_wf = {}

    for wf_name, wf in welfare_functions.items():
        print(f"Running BAU welfare comparison for: {wf_name}")

        JUSTICE.hard_reset()
        model = JUSTICE(
            start_year=START_YEAR,
            end_year=END_YEAR,
            timestep=TIMESTEP,
            scenario=2,
            climate_ensembles=1,
            stochastic_run=False,
            social_welfare_function=wf,
        )

        ecr_bau = np.zeros(
            (
                len(model.data_loader.REGION_LIST),
                len(model.time_horizon.model_time_horizon),
            )
        )

        model.run(emission_control_rate=ecr_bau, endogenous_savings_rate=True)
        data = model.evaluate()

        welfare_value = np.asarray(data["welfare"]).item()
        results_wf[wf_name] = float(welfare_value)

    results_wf_df = pd.DataFrame.from_dict(
        results_wf,
        orient="index",
        columns=["BAU welfare"]
    )

    display(results_wf_df)

else:
    print("Skipping optional BAU welfare comparison.")

Running BAU welfare comparison for: Utilitarian
Running BAU welfare comparison for: Prioritarian
Running BAU welfare comparison for: Sufficientarian


,BAU welfare
Utilitarian,-103.721111
Prioritarian,-414.827668
Sufficientarian,-104.187375


## 2. Build the optimisation setup

**Project purpose:** define the model search problem that the MOEA will solve.

**Main source:** Assignment 4 — RBF structure and optimisation configuration.  
**Output of this step:** `config/config_student.json`.

This is where Assignment 4 fits into the project. It gives the computational definition of the South Africa problem framing.

### 2.1 Adaptive RBF policy design

The policy is not a fixed year-by-year emissions path.

Instead, the RBF maps:

```text
current global temperature + rate of temperature change → regional emission control rates
```

This is important for South Africa because it avoids a rigid equal-obligation pathway and allows policy to adapt to observed climate conditions.



The policy is not represented as a fixed year-by-year emissions path. Instead, JUSTICE uses an adaptive RBF policy rule. The RBF maps the current global temperature and the rate of temperature change to regional emission control rates.

In model terms:

current global temperature + temperature change → RBF policy rule → regional emission control rates

This is relevant for South Africa because it avoids evaluating only rigid equal-obligation pathways. Instead, mitigation can adapt to observed climate conditions. However, the RBF parameters themselves are not interpreted as political goals. They are technical decision variables used by the optimizer to generate candidate policies.

For South Africa, the key analytical step is therefore not to manually change the RBF architecture, but to evaluate the resulting policies using South Africa-specific satisficing criteria. A policy is acceptable only if it limits climate risk while keeping South Africa’s mitigation burden, measured as abatement_cost / gross_economic_output, within an acceptable range.

In [12]:
# Assignment 4 — RBF decision variable structure.

n_rbfs = N_RBFS # dit is dus al ingevuld 
n_inputs = N_INPUTS
n_outputs = N_REGIONS
 

n_centers = n_rbfs * n_inputs
n_radii = n_rbfs * n_inputs
n_weights = n_rbfs * n_outputs
n_total = n_centers + n_radii + n_weights

rbf_structure = pd.DataFrame([
    {"Component": "Centers", "Formula": "n_rbfs * n_inputs", "Count": n_centers},
    {"Component": "Radii", "Formula": "n_rbfs * n_inputs", "Count": n_radii},
    {"Component": "Weights", "Formula": "n_rbfs * n_outputs", "Count": n_weights},
    {"Component": "Total", "Formula": "centers + radii + weights", "Count": n_total},
])

rbf_structure

,Component,Formula,Count
0,Centers,n_rbfs * n_inputs,8
1,Radii,n_rbfs * n_inputs,8
2,Weights,n_rbfs * n_outputs,228
3,Total,centers + radii + weights,244


**Interpretation for report:**

> The RBF structure uses 244 decision variables. These variables encode an adaptive decision rule rather than a fixed mitigation action for every region and every year. This is politically relevant for South Africa because it avoids assuming that all countries must follow the same rigid mitigation pathway regardless of climate response or transition burden.

### 2.2 Welfare function choice

**Source:** Assignment 4, Task 5.1.  
**Project role:** define whose welfare the optimiser prioritises.

South Africa's main optimisation uses **Prioritarian** welfare because the mandate asks for extra weight for worse-off regions. The rival framing later is **Utilitarian**, because that is the obvious global-efficiency counterargument.

In [13]:
WELFARE_OPTIONS = {
    "Utilitarian": WelfareFunction.UTILITARIAN,
    "Prioritarian": WelfareFunction.PRIORITARIAN,
    "Egalitarian": WelfareFunction.EGALITARIAN,
}

print("Welfare functions considered:")
for name, wf in WELFARE_OPTIONS.items():
    print(f"  {name:<20s} internal index = {wf.value[0]}")

CHOSEN_WELFARE_FUNCTION = WELFARE_OPTIONS["Prioritarian"]
CHOSEN_WELFARE_FUNCTION_IDX = CHOSEN_WELFARE_FUNCTION.value[0]

print(f"\nChosen: {CHOSEN_WELFARE_FUNCTION.name} (index {CHOSEN_WELFARE_FUNCTION_IDX})")
print("Rationale: Prioritarian welfare fits South Africa because it gives extra weight to worse-off regions.")

Welfare functions considered:
  Utilitarian          internal index = 0
  Prioritarian         internal index = 1
  Egalitarian          internal index = 3

Chosen: PRIORITARIAN (index 1)
Rationale: Prioritarian welfare fits South Africa because it gives extra weight to worse-off regions.


### 2.3 Reference SSP-RCP scenario

**Source:** Assignment 4, Task 5.2.  
**Project role:** choose the baseline scenario under which the MOEA searches.

Use **SSP2-RCP4.5** as a middle-of-the-road reference scenario. Robustness across other scenarios is tested later.

### 2.1 Adaptive RBF policy design

The policy is not a fixed year-by-year emissions path.

Instead, the RBF maps:

```text
current global temperature + rate of temperature change → regional emission control rates
```

This is important for South Africa because it avoids a rigid equal-obligation pathway and allows policy to adapt to observed climate conditions.

In [14]:
# Inspect scenario options if needed.
scenario_options = [(s.value[0], s.value[3]) for s in Scenario]
pd.DataFrame(scenario_options, columns=["index", "scenario"])

,index,scenario
0,0,SSP1-RCP1.9
1,1,SSP1-RCP2.6
2,2,SSP2-RCP4.5
3,3,SSP3-RCP7.0
4,4,SSP4-RCP3.4
5,5,SSP4-RCP6.0
6,6,SSP5-RCP3.4-overshoot
7,7,SSP5-RCP8.5


In [15]:
CHOSEN_SCENARIO_INDEX = 2
CHOSEN_SCENARIO_NAME = "SSP2-RCP4.5"

print(f"Chosen reference scenario: {CHOSEN_SCENARIO_INDEX} — {CHOSEN_SCENARIO_NAME}")

Chosen reference scenario: 2 — SSP2-RCP4.5


### 2.4 FaIR ensemble strategy

**Source:** Assignment 4, Task 5.3.  
**Project role:** decide how much climate uncertainty to include during local optimisation.

For local runs, use a small reproducible ensemble sample for speed. Later, test robustness with more scenarios or ensemble members.

In [16]:
FULL_ENSEMBLE_SIZE = 1001
N_ENSEMBLE_LOCAL = 10

rng = np.random.default_rng(seed=42)
ensemble_indices = sorted(
    rng.choice(FULL_ENSEMBLE_SIZE, size=N_ENSEMBLE_LOCAL, replace=False).tolist()
)

speedup = FULL_ENSEMBLE_SIZE / N_ENSEMBLE_LOCAL

print("Selected ensemble indices:", ensemble_indices)
print(f"Approximate speedup vs full ensemble: {speedup:.1f}x")

Selected ensemble indices: [85, 88, 94, 201, 431, 436, 650, 696, 768, 856]
Approximate speedup vs full ensemble: 100.1x


For the local optimization run, we use a subset of 15 FaIR ensemble members. This is a pragmatic compromise between computational feasibility and representing climate response uncertainty. Running all 1001 ensemble members would better capture the full FaIR uncertainty range, but would be too expensive for our local workflow. Because we have access to a relatively fast computer, we use 15 members rather than a smaller subset such as 5 or 10.

The members are selected using evenly spaced ensemble indices between 0 and 1000. Since FaIR ensemble indices are identifiers and are not ordered by ECS, this should not be interpreted as evenly sampling the climate sensitivity range. It is a computationally practical way to avoid using only neighbouring ensemble members.

For final interpretation, selected policies should ideally be re-evaluated on a larger subset or, if feasible, on the full 1001-member ensemble.

### 2.5 Objectives and epsilon values

**Source:** Assignment 4, Task 5.4.  
**Project role:** define what the optimiser considers a good policy.

For South Africa, all four objectives are useful because they show the core political tension:

- climate safety,
- welfare,
- damage burden,
- abatement burden.

In [17]:
from ema_workbench import ScalarOutcome

EPSILONS = [
    10.0,   # welfare
    0.01,   # fraction_above_threshold
    10.0,   # welfare_loss_damage
    10.0,   # welfare_loss_abatement
]

objectives =[
    ScalarOutcome("welfare", kind=ScalarOutcome.MINIMIZE),
    ScalarOutcome("fraction_above_threshold", kind=ScalarOutcome.MINIMIZE),
    ScalarOutcome("welfare_loss_damage", kind=ScalarOutcome.MAXIMIZE),
    ScalarOutcome("welfare_loss_abatement", kind=ScalarOutcome.MAXIMIZE),
]

objective_table = pd.DataFrame([
    {
        "Objective": obj.name,
        "Direction": "MINIMIZE" if obj.kind == ScalarOutcome.MINIMIZE else "MAXIMIZE",
        "Epsilon": eps,
        "South Africa interpretation": interp,
    }
    for obj, eps, interp in zip(
        objectives,
        EPSILONS,
        [
            "Lower welfare loss is better.",
            "Lower probability/fraction above 2°C is better.",
            "Assignment convention: larger stored value means less actual damage burden.",
            "Assignment convention: larger stored value means less actual abatement burden.",
        ],
    )
])

objective_table

,Objective,Direction,Epsilon,South Africa interpretation
0,welfare,MINIMIZE,10.00,Lower welfare loss is better.
1,fraction_above_threshold,MINIMIZE,0.01,Lower probability/fraction above 2°C is better.
2,welfare_loss_damage,MAXIMIZE,10.00,Assignment convention: larger stored value mea...
3,welfare_loss_abatement,MAXIMIZE,10.00,Assignment convention: larger stored value mea...


### 2.6 Save Assignment 4 configuration

**Source:** Assignment 4, Task 5.5.  
**Project role:** create the config used by Assignment 5.

This config is the bridge from problem framing to optimisation.

In [18]:
student_config = {
    "start_year": START_YEAR,
    "end_year": END_YEAR,
    "data_timestep": DATA_TIMESTEP,
    "timestep": TIMESTEP,
    "emission_control_start_year": 2025,
    "n_rbfs": N_RBFS,
    "n_inputs": N_INPUTS,
    "epsilons": EPSILONS,
    "temperature_year_of_interest": 2100,
    "reference_ssp_rcp_scenario_index": CHOSEN_SCENARIO_INDEX,

    # These extra keys are useful for newer/local scripts. If a script ignores them, that is fine.
    "stochastic_run": False,
    "climate_ensemble_members": ensemble_indices,
    "social_welfare_function_type": CHOSEN_WELFARE_FUNCTION_IDX,
}

assert student_config["n_rbfs"] == student_config["n_inputs"] + 2
assert len(student_config["epsilons"]) == 4
assert all(e > 0 for e in student_config["epsilons"])
assert student_config["emission_control_start_year"] >= student_config["start_year"]

config_path = _CONFIG_DIR / "config_student.json"
with open(config_path, "w") as fh:
    json.dump(student_config, fh, indent=4)

print(f"Config saved → {config_path}")
print(json.dumps(student_config, indent=4))

Config saved → /Users/wesselmuntendam/Model Based Decision Making/epa141a/config/config_student.json
{
    "start_year": 2015,
    "end_year": 2300,
    "data_timestep": 5,
    "timestep": 1,
    "emission_control_start_year": 2025,
    "n_rbfs": 4,
    "n_inputs": 2,
    "epsilons": [
        10.0,
        0.01,
        10.0,
        10.0
    ],
    "temperature_year_of_interest": 2100,
    "reference_ssp_rcp_scenario_index": 2,
    "stochastic_run": false,
    "climate_ensemble_members": [
        85,
        88,
        94,
        201,
        431,
        436,
        650,
        696,
        768,
        856
    ],
    "social_welfare_function_type": 1
}


## 3. Generate candidate policies through MOEA

**Project purpose:** use optimisation to generate many candidate adaptive mitigation policies.

**Main source:** Assignment 5.  
**Input:** `config_student.json` from Step 2.  
**Output:** Pareto archive files in `results/`.

Interpretation:

> Assignment 5 produces the policy options. It does not yet tell us which policy South Africa should choose.

### 3.1 Local optimisation run

Start with a small test run to ensure everything works. Increase `NFE` later if time allows.

- `NFE` = number of candidate policies tested per seed.
- `SEEDS` = independent optimiser runs.
- `N_ENSEMBLES` = climate ensemble members used during local optimisation.

In [19]:
import warnings
warnings.filterwarnings("ignore")

import os, sys, json
import numpy as np
import pandas as pd

# Locate JUSTICE-main (one level up from model_answers_ema/)
try:
    _NOTEBOOK_DIR = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
except NameError:
    _NOTEBOOK_DIR = os.path.abspath('.')
_JUSTICE_ROOT = os.path.normpath(os.path.join(_NOTEBOOK_DIR, "../JUSTICE-main"))

if _JUSTICE_ROOT not in sys.path:
    sys.path.insert(0, _JUSTICE_ROOT)

os.chdir(_JUSTICE_ROOT)

# ── Inspect config ────────────────────────────────────────────────────────────
CONFIG_PATH = os.path.join(_NOTEBOOK_DIR, "../config/config_student.json")

with open(CONFIG_PATH) as fh:
    cfg = json.load(fh)

explanations = {
    "start_year":                       "Simulation start year",
    "end_year":                         "Simulation end year",
    "data_timestep":                    "Years between raw input data points",
    "timestep":                         "Model integration timestep (years)",
    "emission_control_start_year":      "First year ECR can exceed zero",
    "n_rbfs":                           "Number of RBFs (effective: n_inputs + 2)",
    "n_inputs":                         "RBF input signals ",
    "epsilons":                         "Archive granularity",
    "temperature_year_of_interest":     "Year for threshold fraction evaluation",
    "reference_ssp_rcp_scenario_index": "Reference scenario index",
}

print(f"Configuration: {CONFIG_PATH}\n")
for k, v in cfg.items():
    print(f"  {k:<40s}  {str(v):<15}  # {explanations.get(k, '')}")

Configuration: /Users/wesselmuntendam/Model Based Decision Making/epa141a/JUSTICE-main/../config/config_student.json

  start_year                                2015             # Simulation start year
  end_year                                  2300             # Simulation end year
  data_timestep                             5                # Years between raw input data points
  timestep                                  1                # Model integration timestep (years)
  emission_control_start_year               2025             # First year ECR can exceed zero
  n_rbfs                                    4                # Number of RBFs (effective: n_inputs + 2)
  n_inputs                                  2                # RBF input signals 
  epsilons                                  [10.0, 0.01, 10.0, 10.0]  # Archive granularity
  temperature_year_of_interest              2100             # Year for threshold fraction evaluation
  reference_ssp_rcp_scenario_index         

In [20]:
# # Assignment 5 — local MOEA run.
# # Keep small for a first test. Increase NFE later if runtime allows.
# import shlex
#
# NFE = 10
# SEEDS = [1, 2, 3, 4, 5]
# N_ENSEMBLES = N_ENSEMBLE_LOCAL
# N_PROCESSES = None
# OUTPUT_DIR = RESULTS_DIR
#
# os.makedirs(OUTPUT_DIR, exist_ok=True)
#
# seeds_str = " ".join(str(s) for s in SEEDS)
# n_proc_arg = f"--n_processes {N_PROCESSES}" if N_PROCESSES else ""
#
# # Relative to the notebook folder, so it works on other computers too
# script_path = os.path.abspath(
#     os.path.join(_NOTEBOOK_DIR, "../A- Project G15/run_optimization_local.py")
# )
#
# cmd = (
#     f"python {shlex.quote(script_path)} "
#     f"--nfe {NFE} "
#     f"--seeds {seeds_str} "
#     f"--n_ensembles {N_ENSEMBLES} "
#     f"--output_dir {shlex.quote(str(OUTPUT_DIR))} "
#     f"--config {shlex.quote(str(config_path))} "
#     + n_proc_arg
# )
#
# print("Command to run:")
# print(cmd)
#
# ret = os.system(cmd)
# print(f"\nExit code: {ret} ({'OK' if ret == 0 else 'ERROR'})")

In [21]:
import shlex

NFE = 10
SEEDS = [1, 2, 3, 4, 5]
N_ENSEMBLES = N_ENSEMBLE_LOCAL
N_PROCESSES = None
OUTPUT_DIR = RESULTS_DIR

seeds_str = " ".join(str(s) for s in SEEDS)
n_proc_arg = f"--n_processes {N_PROCESSES}" if N_PROCESSES else ""

script_path = (_PROJECT_DIR / "run_optimization_local.py").resolve()
config_path = (_CONFIG_DIR / "config_student.json").resolve()

cmd = (
    f"python {shlex.quote(str(script_path))} "
    f"--nfe {NFE} "
    f"--seeds {seeds_str} "
    f"--n_ensembles {N_ENSEMBLES} "
    f"--output_dir {shlex.quote(str(OUTPUT_DIR))} "
    f"--config {shlex.quote(str(config_path))} "
    + n_proc_arg
)

print("Command to run:")
print(cmd)

ret = os.system(cmd)
print(f"\nExit code: {ret} ({'OK' if ret == 0 else 'ERROR'})")

Command to run:
python '/Users/wesselmuntendam/Model Based Decision Making/epa141a/A- Project G15/run_optimization_local.py' --nfe 10 --seeds 1 2 3 4 5 --n_ensembles 10 --output_dir '/Users/wesselmuntendam/Model Based Decision Making/epa141a/A- Project G15/results' --config '/Users/wesselmuntendam/Model Based Decision Making/epa141a/config/config_student.json' 
JUSTICE — Local MOEA Optimisation  (Assignment 5)
  Welfare function  : PRIORITARIAN
  Config            : /Users/wesselmuntendam/Model Based Decision Making/epa141a/config/config_student.json
  NFE per seed      : 10
  Seeds (5)        : [1, 2, 3, 4, 5]
  FAIR ensembles    : 10  (indices ≈ [np.int64(1), np.int64(112), np.int64(223), np.int64(334), np.int64(445), np.int64(556), np.int64(667), np.int64(778), np.int64(889), np.int64(1000)])
  Population size   : 100
  Worker processes  : auto
  Output directory  : /Users/wesselmuntendam/Model Based Decision Making/epa141a/A- Project G15/results

[1/5] seed = 1
--------------------

[MainProcess/INFO] pool started with 10 workers
100it [00:06, 16.42it/s]                                                       
[MainProcess/INFO] optimization completed, found 23 solutions
[MainProcess/INFO] terminating pool


    convergence metrics  →  /Users/wesselmuntendam/Model Based Decision Making/epa141a/A- Project G15/results/PRIORITARIAN_10_1/convergence_1.csv
    23 Pareto solutions  →  /Users/wesselmuntendam/Model Based Decision Making/epa141a/A- Project G15/results/PRIORITARIAN_10_1/pareto_front_1.csv
    convergence archive  →  /Users/wesselmuntendam/Model Based Decision Making/epa141a/A- Project G15/results/PRIORITARIAN_10_1/PRIORITARIAN_10_1.tar.gz
    elapsed: 0.3 min

[2/5] seed = 2
----------------------------------------
    workers = auto (cpu_count-1)


[MainProcess/INFO] pool started with 10 workers
100it [00:06, 15.76it/s]                                                       
[MainProcess/INFO] optimization completed, found 18 solutions
[MainProcess/INFO] terminating pool


    convergence metrics  →  /Users/wesselmuntendam/Model Based Decision Making/epa141a/A- Project G15/results/PRIORITARIAN_10_2/convergence_2.csv
    18 Pareto solutions  →  /Users/wesselmuntendam/Model Based Decision Making/epa141a/A- Project G15/results/PRIORITARIAN_10_2/pareto_front_2.csv
    convergence archive  →  /Users/wesselmuntendam/Model Based Decision Making/epa141a/A- Project G15/results/PRIORITARIAN_10_2/PRIORITARIAN_10_2.tar.gz
    elapsed: 0.3 min

[3/5] seed = 3
----------------------------------------
    workers = auto (cpu_count-1)


...

    convergence metrics  →  /Users/wesselmuntendam/Model Based Decision Making/epa141a/A- Project G15/results/PRIORITARIAN_10_3/convergence_3.csv
    15 Pareto solutions  →  /Users/wesselmuntendam/Model Based Decision Making/epa141a/A- Project G15/results/PRIORITARIAN_10_3/pareto_front_3.csv
    convergence archive  →  /Users/wesselmuntendam/Model Based Decision Making/epa141a/A- Project G15/results/PRIORITARIAN_10_3/PRIORITARIAN_10_3.tar.gz
    elapsed: 0.3 min

[4/5] seed = 4
----------------------------------------
    workers = auto (cpu_count-1)


[MainProcess/INFO] pool started with 10 workers
100it [00:06, 16.31it/s]                                                       
[MainProcess/INFO] optimization completed, found 14 solutions
[MainProcess/INFO] terminating pool


    convergence metrics  →  /Users/wesselmuntendam/Model Based Decision Making/epa141a/A- Project G15/results/PRIORITARIAN_10_4/convergence_4.csv
    14 Pareto solutions  →  /Users/wesselmuntendam/Model Based Decision Making/epa141a/A- Project G15/results/PRIORITARIAN_10_4/pareto_front_4.csv
    convergence archive  →  /Users/wesselmuntendam/Model Based Decision Making/epa141a/A- Project G15/results/PRIORITARIAN_10_4/PRIORITARIAN_10_4.tar.gz
    elapsed: 0.3 min

[5/5] seed = 5
----------------------------------------
    workers = auto (cpu_count-1)


[MainProcess/INFO] pool started with 10 workers
100it [00:07, 14.21it/s]                                                       
[MainProcess/INFO] optimization completed, found 17 solutions
[MainProcess/INFO] terminating pool


    convergence metrics  →  /Users/wesselmuntendam/Model Based Decision Making/epa141a/A- Project G15/results/PRIORITARIAN_10_5/convergence_5.csv
    17 Pareto solutions  →  /Users/wesselmuntendam/Model Based Decision Making/epa141a/A- Project G15/results/PRIORITARIAN_10_5/pareto_front_5.csv
    convergence archive  →  /Users/wesselmuntendam/Model Based Decision Making/epa141a/A- Project G15/results/PRIORITARIAN_10_5/PRIORITARIAN_10_5.tar.gz
    elapsed: 0.3 min

All 5 seeds done in 2 min (0.03 h).
Results → /Users/wesselmuntendam/Model Based Decision Making/epa141a/A- Project G15/results

Exit code: 0 (OK)


### 3.2 Policy types expected from the Pareto archive

Later, when you inspect the Pareto archive, classify policies into three intuitive types:

| Policy type | Climate risk | Abatement burden | South Africa meaning |
|---|---:|---:|---|
| A — climate-safe | Low | High | Acceptable only with substantial transition finance |
| B — economically light | Higher | Low | Protects short-term costs but may fail climate responsibility |
| C — compromise | Medium/low | Medium | Likely best negotiating position for South Africa |

This classification is not an official model output. It is your interpretation of the Pareto trade-off.

## 4. Check search quality and build a reference set

**Project purpose:** avoid basing advice on one random optimiser run.

**Main source:** Assignment 6 — convergence and reference set.  
**Input:** Assignment 5 archives from multiple seeds.  
**Output:** combined reference set and convergence evidence.

This is the step that was easy to lose, but it is important:

> Assignment 6 checks whether the search sufficiently explored the policy space.

For the report, this is mostly a **credibility step**, not the main political argument.

### 4.1 Load Assignment 5/6 outputs

Update paths if your Assignment 6 notebook saves files under different names.

In [22]:
# Possible output paths. Adjust these names to match Assignment 6.
REFERENCE_SET_PATHS = [
    RESULTS_DIR / "combined_reference_set.csv",
    RESULTS_DIR / "reference_set.csv",
    RESULTS_DIR / "epsilon_nondominated_set.csv",
]

reference_set = None
for path in REFERENCE_SET_PATHS:
    if path.exists():
        reference_set = pd.read_csv(path)
        print("Loaded reference set:", path)
        break

if reference_set is None:
    print("No reference set found yet. Complete Assignment 6, then update REFERENCE_SET_PATHS if needed.")
else:
    print("Reference set shape:", reference_set.shape)
    display(reference_set.head())

No reference set found yet. Complete Assignment 6, then update REFERENCE_SET_PATHS if needed.


### 4.2 What to take from Assignment 6

Keep these outputs for the project notebook/report:

1. **Convergence plot or epsilon progress plot**  
   Shows whether the search kept improving or flattened out.

2. **Number of solutions in the combined reference set**  
   Shows how many non-dominated candidate policies are available.

3. **Short interpretation**  
   Example:

> The search was run with multiple random seeds and pooled into a combined reference set. This reduces dependence on one stochastic optimiser trajectory. Because the run is local and computationally limited, the reference set is approximate, but sufficient for comparing South Africa-relevant trade-offs.

In [23]:
# Placeholder for Assignment 6 convergence results.
# Replace with the real file/plot from assignment_06_moea_convergence.ipynb.

CONVERGENCE_NOTES = {
    "status": "to be filled after Assignment 6",
    "what_to_record": [
        "number of seeds completed",
        "number of reference set solutions",
        "whether epsilon progress flattened",
        "whether any seed clearly underperformed",
    ],
}

CONVERGENCE_NOTES

{'status': 'to be filled after Assignment 6',
 'what_to_record': ['number of seeds completed',
  'number of reference set solutions',
  'whether epsilon progress flattened',
  'whether any seed clearly underperformed']}

## 5. Understand trade-offs and select candidate policies

**Project purpose:** turn the Pareto archive into policy insight.

**Main source:** Assignment 7 — Pareto visualisation.  
**Input:** combined reference set from Assignment 6.  
**Output:** trade-off plots and selected policy candidates A/B/C.

This is where the results start becoming politically useful for South Africa.

### 5.1 Pareto trade-off plots

The most relevant trade-offs for South Africa are:

1. `fraction_above_threshold` vs `welfare_loss_abatement`
2. `welfare_loss_damage` vs `welfare_loss_abatement`
3. `welfare` vs `fraction_above_threshold`

Use these plots to show that climate safety and transition burden are in tension.

In [24]:
if reference_set is not None:
    required = {"fraction_above_threshold", "welfare_loss_abatement"}
    if required.issubset(reference_set.columns):
        ax = reference_set.plot.scatter(
            x="fraction_above_threshold",
            y="welfare_loss_abatement",
            figsize=(7, 5),
            title="Pareto trade-off: climate threshold risk vs abatement burden metric",
        )
        ax.set_xlabel("Fraction above temperature threshold")
        ax.set_ylabel("Welfare loss abatement metric")
        plt.show()
    else:
        print("Reference set loaded, but expected columns are missing:", required - set(reference_set.columns))
else:
    print("No reference set loaded yet. Run Assignment 6 first.")

No reference set loaded yet. Run Assignment 6 first.


### 5.2 Select policy A, B and C

After inspecting Pareto plots, select three policies:

- **Policy A:** most climate-safe / lowest `fraction_above_threshold`
- **Policy B:** lowest transition burden / best abatement-burden score
- **Policy C:** compromise policy for South Africa

Policy C is likely the one you will re-evaluate and defend.

In [25]:
# Simple template for selecting policy candidates.
# Adjust column names/directions once you inspect your actual reference_set.

selected_policies = {}

if reference_set is not None and len(reference_set) > 0:
    if "fraction_above_threshold" in reference_set.columns:
        selected_policies["A_climate_safe"] = reference_set["fraction_above_threshold"].idxmin()

    if "welfare_loss_abatement" in reference_set.columns:
        # Following assignment convention: larger stored value can mean lower actual burden.
        selected_policies["B_economically_light"] = reference_set["welfare_loss_abatement"].idxmax()

    # Compromise placeholder: choose a central solution after visual inspection.
    # A better choice can use normalized distance to ideal point.
    selected_policies["C_compromise_placeholder"] = reference_set.index[len(reference_set) // 2]

selected_policies

{}

### 5.3 Interpretation template after Assignment 7

Fill in after the Pareto plots are made:

> The Pareto front shows that policies with lower temperature-threshold risk tend to require a higher abatement effort. For South Africa, this means that ambitious climate action is not rejected, but it must be paired with financial support because the transition burden is politically and economically relevant.

## 6. Re-evaluate preferred policy for full model time series

**Project purpose:** get full outputs for the chosen South Africa policy.

**Main source:** Assignment 7/8 plus extra re-evaluation.  
**Input:** selected preferred policy from the Pareto archive.  
**Output:** full time series: temperature, ECR, abatement costs, damages, output, consumption.

The Pareto archive usually contains summary objective values. For South Africa, you need full regional time series to compute:

```python
zaf_abatement_burden = zaf_abatement_cost / zaf_gross_economic_output
```

In [26]:
# Re-evaluation placeholder.
# The exact implementation depends on how Assignment 7/8 stores the preferred policy weights.

PREFERRED_POLICY_INDEX = selected_policies.get("C_compromise_placeholder", None) if "selected_policies" in globals() else None
PREFERRED_POLICY_DATA_PATH = RESULTS_DIR / "preferred_policy_reevaluation.npy"

print("Preferred policy index:", PREFERRED_POLICY_INDEX)
print("Expected re-evaluation output path:", PREFERRED_POLICY_DATA_PATH)
print("After re-evaluation, load the full JUSTICE datasets into a variable called preferred_data.")

preferred_data = None
if PREFERRED_POLICY_DATA_PATH.exists():
    preferred_data = np.load(PREFERRED_POLICY_DATA_PATH, allow_pickle=True).item()
    print("Loaded preferred policy re-evaluation data.")
else:
    print("No re-evaluation data found yet. Complete the re-evaluation step before Section 7.")

Preferred policy index: None
Expected re-evaluation output path: /Users/wesselmuntendam/Model Based Decision Making/epa141a/A- Project G15/results/preferred_policy_reevaluation.npy
After re-evaluation, load the full JUSTICE datasets into a variable called preferred_data.
No re-evaluation data found yet. Complete the re-evaluation step before Section 7.


### 6.1 What to extract from re-evaluation

For your preferred policy, keep:

| Variable | Why it matters for South Africa |
|---|---|
| `global_temperature` | Shows climate safety |
| `emission_control_rate` or `constrained_emission_control_rate` | Shows South Africa's mitigation pathway |
| `abatement_cost` | Transition cost |
| `gross_economic_output` | Denominator for relative burden |
| `economic_damage` | Cost of insufficient mitigation |
| `consumption_per_capita` | Welfare-relevant living standard |

## 7. South Africa-specific burden analysis

**Project purpose:** convert model outputs into the main South Africa argument.

**Main source:** re-evaluation outputs from Step 6.  
**Supporting source:** South Africa mandate from Assignment 4.  
**Output:** tables/figures comparing `zaf` burden with developed and peer regions.

This is probably the most important project-specific section.

### 7.1 Compute South Africa burden metrics

The key indicator is:

```python
abatement_cost / gross_economic_output
```

This expresses transition cost as a share of the economy.

In [27]:
def extract_region_burden_metrics(data, model, region_code):
    # Compute simple regional burden metrics from a JUSTICE output dictionary.
    idx = list(model.data_loader.REGION_LIST).index(region_code)

    gross_output = data["gross_economic_output"][idx, :, :]
    abatement_cost = data["abatement_cost"][idx, :, :]
    economic_damage = data["economic_damage"][idx, :, :]
    consumption_pc = data["consumption_per_capita"][idx, :, :]

    abatement_burden = np.divide(
        abatement_cost,
        gross_output,
        out=np.full_like(abatement_cost, np.nan, dtype=float),
        where=gross_output != 0,
    )
    damage_burden = np.divide(
        economic_damage,
        gross_output,
        out=np.full_like(economic_damage, np.nan, dtype=float),
        where=gross_output != 0,
    )

    return {
        "region": region_code,
        "mean_abatement_burden": float(np.nanmean(abatement_burden)),
        "max_abatement_burden": float(np.nanmax(abatement_burden)),
        "mean_damage_burden": float(np.nanmean(damage_burden)),
        "max_damage_burden": float(np.nanmax(damage_burden)),
        "mean_consumption_per_capita": float(np.nanmean(consumption_pc)),
    }

In [28]:
# If preferred_data is available, compute South Africa metrics.
# If not, this cell will wait until re-evaluation is done.

if preferred_data is not None:
    zaf_metrics = extract_region_burden_metrics(preferred_data, _model_probe, "zaf")
    zaf_metrics
else:
    print("preferred_data is not loaded yet. Complete Section 6 re-evaluation first.")

preferred_data is not loaded yet. Complete Section 6 re-evaluation first.


### 7.2 Compare South Africa with peer and developed regions

Use this to support the claim:

> Equal mitigation obligations are not equally fair if they create higher costs relative to GDP for coal-dependent emerging economies.

In [29]:
compare_regions = [
    # South Africa and African/regional peers
    "zaf", "rsaf", "egy", "noan", "noap",

    # Emerging/transition economies
    "pol", "tur", "mex", "bra", "idn", "chn", "rus",

    # Developed reference countries/regions
    "usa", "gbr", "fra", "rfa",
]

if preferred_data is not None:
    rows = []
    for region in compare_regions:
        if region in REGION_LIST:
            try:
                rows.append(extract_region_burden_metrics(preferred_data, _model_probe, region))
            except Exception as err:
                print(f"Skipping {region}: {err}")
        else:
            print(f"Region code not found in model region list: {region}")

    burden_comparison = pd.DataFrame(rows)
    display(burden_comparison.sort_values("mean_abatement_burden", ascending=False))
else:
    print("preferred_data is not loaded yet. Complete Section 6 re-evaluation first.")

preferred_data is not loaded yet. Complete Section 6 re-evaluation first.


### 7.3 Interpretation template

Fill in after the table is available:

> South Africa's abatement burden reaches **[X]%** of gross economic output, compared with **[Y]%** for **[developed country/region]**. This supports South Africa's argument that climate obligations should not be evaluated only by aggregate emissions or global welfare. When costs are expressed relative to economic capacity, the transition burden is politically unequal.

If results do not show a higher burden, use this interpretation instead:

> The model does not strongly show a higher average burden for South Africa under this policy. This weakens the strongest version of the finance claim, but the mandate still supports transition finance because the model does not represent domestic labour disruption and coal-community impacts.

## 8. Robustness and satisficing

**Project purpose:** test whether the preferred policy remains acceptable under uncertainty.

**Main source:** Assignment 8.  
**Input:** preferred policy and uncertainty/scenario re-evaluations.  
**Output:** fraction of cases where South Africa's satisficing criterion is met.

Question:

> Does the preferred policy remain acceptable across different SSP-RCP scenarios and climate ensemble members?

In [30]:
ROBUSTNESS_PATHS = [
    RESULTS_DIR / "robustness_results.csv",
    RESULTS_DIR / "satisficing_results.csv",
]

robustness = None
for path in ROBUSTNESS_PATHS:
    if path.exists():
        robustness = pd.read_csv(path)
        print("Loaded robustness results:", path)
        break

if robustness is None:
    print("Robustness results not found yet. Complete Assignment 8, then update ROBUSTNESS_PATHS if needed.")
else:
    display(robustness.head())

Robustness results not found yet. Complete Assignment 8, then update ROBUSTNESS_PATHS if needed.


### 8.1 Robustness interpretation template

Fill in after Assignment 8:

> The preferred South Africa compromise policy satisfies the climate-and-transition criterion in **[X]%** of re-evaluation cases. This means the policy is **[robust / partly robust / not robust]** under the tested uncertainty set.

Possible conclusions:

| Robustness result | Interpretation for South Africa |
|---|---|
| High climate and burden robustness | Policy C is defensible as South Africa's preferred agreement position |
| Good climate robustness but poor burden robustness | Strong case for Just Transition finance |
| Good burden robustness but poor climate robustness | Policy is too weak for African climate vulnerability |
| Low robustness on both | Need adaptive triggers or stronger finance/technology assumptions |

## 9. Rival framing: Prioritarian vs Utilitarian

**Project purpose:** show that model advice depends on value assumptions.

**Source:** recommended by the universal workflow; can be done as extra re-optimisation or simpler re-evaluation.  
**Input:** preferred South Africa policy and/or utilitarian Pareto archive.  
**Output:** comparison of South Africa-preferred framing versus global-total framing.

Why this matters:

> South Africa's key counterargument is that a global total welfare framing can hide unequal transition burdens.

### 9.1 Two ways to do rival framing

**Best option, if compute time allows:**

Re-run Assignment 5 under `WelfareFunction.UTILITARIAN`, then compare Pareto fronts:

```text
Prioritarian Pareto front vs Utilitarian Pareto front
```

**Minimum option, if compute time is limited:**

Evaluate the chosen South Africa policy under a utilitarian welfare lens and compare whether its ranking/conclusion changes.

In [31]:
RIVAL_FRAMING_NOTES = {
    "preferred_framing": "Prioritarian",
    "rival_framing": "Utilitarian",
    "minimum_analysis": "Re-evaluate preferred policy under utilitarian welfare if re-optimisation is too slow.",
    "full_analysis": "Run a second MOEA under utilitarian welfare and compare Pareto fronts.",
}

RIVAL_FRAMING_NOTES

{'preferred_framing': 'Prioritarian',
 'rival_framing': 'Utilitarian',
 'minimum_analysis': 'Re-evaluate preferred policy under utilitarian welfare if re-optimisation is too slow.',
 'full_analysis': 'Run a second MOEA under utilitarian welfare and compare Pareto fronts.'}

### 9.2 Rival framing interpretation template

Fill in after rival framing analysis:

> Under the prioritarian framing, policies that reduce unequal burdens are valued more strongly. Under the utilitarian framing, the ranking shifts toward policies that perform well in aggregate. This shows that the model result is not purely technical; it depends on whose welfare is prioritised.

Debate line:

> A global total welfare framing can make equal obligations appear efficient, but a worst-off-weighted framing reveals that the same obligations may create unequal burdens.

## 10. Final policy advice for South Africa

**Project purpose:** translate the model-based analysis into a clear negotiating position.

This is not a new computation step. It combines everything above.

### 10.1 Argument chain

Use this as the backbone of the final report:

1. **Problem:** South Africa must decarbonise, but coal dependence makes rapid transition costly.
2. **Framing:** Use prioritarian welfare and measure burden as share of output.
3. **Search:** Use MOEA to generate adaptive RBF mitigation policies.
4. **Quality check:** Use convergence/reference set to avoid relying on one random run.
5. **Trade-off:** Pareto front shows climate safety versus transition burden.
6. **Policy choice:** Choose compromise policy C, not purely cheapest or purely strictest.
7. **Re-evaluation:** Extract full time series for the preferred policy.
8. **Burden analysis:** Compare `zaf` abatement burden with developed regions.
9. **Robustness:** Check whether the policy remains acceptable under uncertainty.
10. **Rival framing:** Show that utilitarian/global-total framing can hide distributional burdens.
11. **Advice:** Support ambitious mitigation only with dedicated Just Transition finance.

### 10.2 Draft policy advice

> South Africa should support an ambitious but adaptive mitigation agreement, conditional on a dedicated Just Transition fund for coal-dependent emerging economies. The model-based analysis should not only evaluate global welfare or aggregate emissions reductions, but also the relative transition burden faced by South Africa. If abatement costs are high relative to gross economic output, equal mitigation obligations become unequal in practice. A prioritarian welfare framing and South Africa-specific burden analysis therefore support the position that climate ambition and transition finance must be negotiated together.

## 11. Figures and tables checklist for the report

Use this checklist to decide what should actually appear in the final report.

| Figure/table | Project step | Assignment source | Why it matters |
|---|---|---|---|
| XLRM table | Step 1 | Assignment 4 | Shows problem framing |
| RBF decision variable table | Step 2 | Assignment 4 | Explains adaptive policy design |
| Optimisation config table | Step 2 | Assignment 4 | Documents welfare, scenario, objectives |
| Convergence/reference set evidence | Step 4 | Assignment 6 | Shows search credibility |
| Pareto front plot | Step 5 | Assignment 7 | Shows trade-off between climate risk and burden |
| Policy A/B/C comparison | Step 5 | Assignment 7 | Justifies preferred policy selection |
| Time series of preferred policy | Step 6 | Re-evaluation | Shows how policy develops over time |
| `zaf` burden comparison table | Step 7 | Re-evaluation / own analysis | Main evidence for South Africa mandate |
| Robustness/satisficing table | Step 8 | Assignment 8 | Tests whether advice holds under uncertainty |
| Rival framing comparison | Step 9 | Extra / recommended | Shows values shape model advice |

## 12. Limitations and discussion notes

Potential limitations to discuss:

1. **Local optimisation is computationally limited.** The Pareto front is approximate.
2. **Small ensemble samples may underrepresent climate uncertainty.** Robustness analysis partly addresses this.
3. **GDP share is useful but incomplete.** It does not fully capture labour disruption, coal communities, or political feasibility.
4. **JUSTICE abstracts from domestic distribution.** South African coal workers and communities are not explicitly modelled.
5. **Welfare functions are normative choices.** Prioritarian framing is justified by the mandate, but rival actors may reject it.
6. **Finance is not fully endogenous.** The Just Transition fund is treated as a policy condition, not directly modelled unless a cost-reduction/backstop-cost scenario is added.

## 13. One-page working summary

**Research question**  
What adaptive mitigation policy can South Africa support while avoiding a disproportionate transition burden?

**Method**  
We use JUSTICE to optimise adaptive RBF mitigation policies under a prioritarian welfare framing and SSP2-RCP4.5 reference scenario. We combine MOEA runs into a reference set, visualise trade-offs, choose a compromise policy, re-evaluate it for full time series, test robustness, and compare against a utilitarian rival framing.

**Main metric**  
`abatement_cost / gross_economic_output` for `zaf`.

**Expected advice**  
South Africa should support ambitious mitigation only if it is paired with dedicated Just Transition finance.

**Debate line**  
Equal obligations are not equal fairness when transition costs are unequal relative to GDP.